In [1]:
!pip install transformers datasets rouge-score sentencepiece accelerate -q

  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'None'}")

Device: cuda
GPU: Tesla T4


In [3]:
df = pd.read_csv("train_augmented.csv")         # ← point to your cleaned CSV

# Keep only rows where both columns are non-null and non-empty
df = df[["chunk_text", "summary_text"]].dropna()
df = df[df["chunk_text"].str.strip() != ""]
df = df[df["summary_text"].str.strip() != ""]
df = df.reset_index(drop=True)

print(f"Total rows        : {len(df)}")
print(f"Avg input length  : {df['chunk_text'].str.split().str.len().mean():.0f} words")
print(f"Avg target length : {df['summary_text'].str.split().str.len().mean():.0f} words")
df.head(2)

Total rows        : 381
Avg input length  : 342 words
Avg target length : 167 words


,chunk_text,summary_text
0,Vivad se Vishwas I Relief for MSMEs . In cases...,Vivad se Vishwas I will provide relief to MSME...
1,20 crore of central tax and to reduce the amou...,The proposed GST amendments further strengthen...


In [4]:
#train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
#val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42)
train_df=df.copy()
val_df=pd.read_csv("val.csv")
test_df=pd.read_csv("test.csv")
print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

# Convert to HuggingFace Dataset
def to_hf_dataset(dataframe):
    return Dataset.from_dict({
        "input_text" : dataframe["chunk_text"].tolist(),
        "target_text": dataframe["summary_text"].tolist(),
    })

dataset = DatasetDict({
    "train": to_hf_dataset(train_df),
    "validation": to_hf_dataset(val_df),
    "test": to_hf_dataset(test_df),
})
print(dataset)

Train : 381
Val   : 16
Test  : 16
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 381
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 16
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 16
    })
})


In [5]:
MODEL_CKPT = "facebook/bart-base"

# These lengths are based on your data stats (avg ~340 words input, ~210 words summary)
# bart-base max is 1024 — we cap input at 1024 and output at 256
MAX_INPUT_LEN  = 1024
MAX_TARGET_LEN = 256

tokenizer = BartTokenizer.from_pretrained(MODEL_CKPT)

def preprocess(batch):
    # Tokenize inputs
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,        # padding done dynamically by DataCollator
    )
    # Tokenize targets — use text_target for BART
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=["input_text", "target_text"],
)
print(tokenized)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/381 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 381
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 16
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 16
    })
})


In [6]:
model = BartForConditionalGeneration.from_pretrained(MODEL_CKPT)
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Total params    : 139.4M
Trainable params: 139.4M


In [7]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,     # speeds up on Tensor cores
)

In [8]:
!pip install bert-score -q

from bert_score import score as bert_score_fn

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If model returns logits (3D) instead of token ids (2D), take argmax
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    vocab_size = tokenizer.vocab_size

    # Clip + cast — fast tokenizer (Rust) overflows on int64 or out-of-range ids
    predictions = np.clip(predictions, 0, vocab_size - 1).astype(np.int32)
    labels      = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels      = np.clip(labels, 0, vocab_size - 1).astype(np.int32)

    # Decode row by row to isolate any remaining bad tokens
    decoded_preds, decoded_labels = [], []
    for pred_ids, label_ids in zip(predictions.tolist(), labels.tolist()):
        try:
            decoded_preds.append(tokenizer.decode(pred_ids, skip_special_tokens=True).strip())
        except Exception:
            decoded_preds.append("")
        try:
            decoded_labels.append(tokenizer.decode(label_ids, skip_special_tokens=True).strip())
        except Exception:
            decoded_labels.append("")

    # ── ROUGE ────────────────────────────────────────────────────────────────
    r1 = r2 = rl = 0.0
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = scorer.score(label, pred)
        r1 += scores["rouge1"].fmeasure
        r2 += scores["rouge2"].fmeasure
        rl += scores["rougeL"].fmeasure

    n = len(decoded_preds)

    # ── BERTScore ─────────────────────────────────────────────────────────────
    # distilbert-base-uncased — lightweight, fast on Colab T4
    # Returns Precision, Recall, F1 tensors per sample — we report mean F1
    P, R, F1 = bert_score_fn(
        decoded_preds,
        decoded_labels,
        lang="en",
        model_type="distilbert-base-uncased",
        device=device,
        verbose=False,
    )

    return {
        "rouge1"       : round(r1 / n, 4),
        "rouge2"       : round(r2 / n, 4),
        "rougeL"       : round(rl / n, 4),
        "bertscore_P"  : round(P.mean().item(), 4),
        "bertscore_R"  : round(R.mean().item(), 4),
        "bertscore_F1" : round(F1.mean().item(), 4),
    }

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00


In [9]:
BATCH_SIZE     = 4     # increase to 8 if no OOM
GRAD_ACCUM     = 4     # effective batch = 4 × 4 = 16
LR             = 5e-5
EPOCHS         = 10
OUTPUT_DIR     = "./bart-budget-summarizer"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=0.1,               # 10% of steps for LR warmup
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    # Seq2Seq specific
    predict_with_generate=True,     # use .generate() during eval, not teacher forcing
    generation_max_length=MAX_TARGET_LEN,

    # Eval and saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    # Colab memory optimisation
    fp16=True,                      # mixed precision — halves VRAM usage on T4
    dataloader_num_workers=2,

    # Logging
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",               # set to "wandb" if you want experiment tracking
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bertscore P,Bertscore R,Bertscore F1
1,No log,2.287005,0.447100,0.206800,0.295800,0.844900,0.805000,0.823400
2,No log,2.088310,0.547500,0.269800,0.358000,0.838800,0.859600,0.848900
3,12.412128,1.940488,0.546900,0.269100,0.364800,0.849800,0.857400,0.853400
4,12.412128,1.910289,0.541700,0.264200,0.356800,0.843300,0.856500,0.849700
5,8.807876,1.915236,0.553100,0.272500,0.357100,0.849400,0.858800,0.853900


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=120, training_loss=10.087928517659504, metrics={'train_runtime': 260.6332, 'train_samples_per_second': 14.618, 'train_steps_per_second': 0.921, 'total_flos': 537901881753600.0, 'train_loss': 10.087928517659504, 'epoch': 5.0})

In [11]:
results = trainer.evaluate(tokenized["test"])
print("\nTest Set Results:")
for k, v in results.items():
    print(f"  {k:<30} {v}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Test Set Results:
  eval_loss                      1.8354164361953735
  eval_rouge1                    0.5948
  eval_rouge2                    0.3064
  eval_rougeL                    0.3901
  eval_bertscore_P               0.8655
  eval_bertscore_R               0.8562
  eval_bertscore_F1              0.8608
  eval_runtime                   41.405
  eval_samples_per_second        0.386
  eval_steps_per_second          0.097
  epoch                          5.0


In [13]:
model.save_pretrained(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")
print(f"Model saved to {OUTPUT_DIR}/final")

from google.colab import drive
drive.mount('/content/drive')
!cp -r {OUTPUT_DIR}/final /content/drive/MyDrive/bart-budget-summarizer

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./bart-budget-summarizer/final
Mounted at /content/drive


In [14]:
output_file_path = f"{OUTPUT_DIR}/final/evaluation_results.txt"
with open(output_file_path, "w") as f:
    f.write("Test Set Results:\n")
    for k, v in results.items():
        f.write(f"  {k:<30} {v}\n")
print(f"Evaluation results saved to {output_file_path}")

Evaluation results saved to ./bart-budget-summarizer/final/evaluation_results.txt


In [15]:

def summarize(text: str, max_length: int = 300, min_length: int = 80) -> str:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_INPUT_LEN,
        truncation=True,
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,            # beam search — better quality than greedy
        max_length=max_length,
        min_length=min_length,
        length_penalty=3.0,     # >1 encourages longer summaries
        early_stopping=False,
        no_repeat_ngram_size=3, # prevents repetitive phrases
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


# Test on a row from test set
sample_text = test_df.iloc[0]["chunk_text"]
actual      = test_df.iloc[0]["summary_text"]
predicted   = summarize(sample_text)

print(f"\nINPUT   : {sample_text[:300]}...\n")
print(f"ACTUAL  : {actual}\n")
print(f"PREDICTED: {predicted}")


INPUT   : India is a global leader in software development services, IT enabled services, knowledge process outsourcing services and contract R&D services relating to software development. These business segments are quite inter-connected with each other. All these services are proposed to be clubbed under a ...

ACTUAL  : The Budget proposed consolidating software development, IT enabled services, knowledge process outsourcing and contract R&D services under a single category of Information Technology Services with a common safe harbour margin of 15.5%. The threshold for availing safe harbour was proposed to increase from INR 300 crore to INR 2,000 crore, with approvals through an automated rule-driven process and validity for up to 5 years. The Government also proposed fast-tracking unilateral Advance Pricing Agreements for IT services within 2 years, extendable by 6 months, and extending modified return facilities to associated entities. To attract global investment, a tax holiday 